# Train a attU-Net on ARCADE (artery or stenosis segmentation)

**Before you start**
1. Upload `arcade.zip` to your Google Drive (My Drive).
2. In Colab: **Runtime > Change runtime type > T4 GPU**.
3. Run the cells one by one, top to bottom.

Set `TASK` in the config cell: `syntax` trains artery segmentation, `stenosis` trains stenosis segmentation.
Every polygon in an image is merged into one binary mask (foreground = artery or stenosis).

In [ ]:
import os, zipfile, torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

if not os.path.exists('/content/arcade'):
    !wget -q -O /content/arcade_data.zip "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
    with zipfile.ZipFile('/content/arcade_data.zip') as z:
        z.extractall('/content/arcade')

DATA_DIR = None
for root, dirs, files in os.walk('/content/arcade'):
    if 'syntax' in dirs and 'stenosis' in dirs:
        DATA_DIR = root
        break
print('Data folder:', DATA_DIR)

GPU available: True
Tesla T4
Data folder: /content/arcade/arcade
Data folder: /content/arcade/arcade


In [ ]:
!wget -O /content/arcade.zip "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
!ls -lh /content/arcade.zip

--2026-10-27 15:01:19--  https://zenodo.org/records/10390295/files/arcade.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.47.192, 188.185.42.113, 188.185.45.63, ...
Connecting to zenodo.org (zenodo.org)|188.185.47.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 451573813 (431M) [application/octet-stream]
Saving to: ‘/content/arcade.zip’

/content/arcade.zip 100%[===================>] 430.65M  2.53MB/s    in 3m 51s  

2026-10-27 15:05:11 (1.87 MB/s) - ‘/content/arcade.zip’ saved [451573813/451573813]

-rw-r--r-- 1 root root 431M Oct 27 15:05 /content/arcade.zip


In [ ]:
!ls -lh /content/arcade.zip


-rw-r--r-- 1 root root 431M Oct 27 15:05 /content/arcade.zip


In [ ]:
TASK = 'stenosis'      # 'syntax' = artery segmentation, 'stenosis' = stenosis segmentation
IMG_SIZE = 256       # images are 512x512; 256 trains faster
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-3
THRESHOLD = 0.5
SEED = 42

In [ ]:
import json, random
import numpy as np
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class ArcadeBinary(Dataset):
    """Binary mask = all annotated polygons of an image drawn as foreground."""

    def __init__(self, split, train=False):
        base = os.path.join(DATA_DIR, TASK, split)
        ann_dir = os.path.join(base, 'annotations')
        ann_file = [f for f in os.listdir(ann_dir) if f.endswith('.json')][0]
        with open(os.path.join(ann_dir, ann_file), encoding='utf-8') as f:
            coco = json.load(f)
        self.img_dir = os.path.join(base, 'images')
        self.images = coco['images']
        self.polys = {}
        for a in coco['annotations']:
            seg = a.get('segmentation')
            if isinstance(seg, list):
                self.polys.setdefault(a['image_id'], []).extend(seg)
        self.train = train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        info = self.images[i]
        img = Image.open(os.path.join(self.img_dir, os.path.basename(info['file_name']))).convert('L')
        mask = Image.new('L', img.size, 0)
        draw = ImageDraw.Draw(mask)
        for poly in self.polys.get(info['id'], []):
            pts = [(poly[k], poly[k + 1]) for k in range(0, len(poly) - 1, 2)]
            if len(pts) >= 3:
                draw.polygon(pts, fill=1)
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
        x = np.asarray(img, dtype=np.float32) / 255.0
        y = np.asarray(mask, dtype=np.float32)
        if self.train and random.random() < 0.5:      # simple augmentation: horizontal flip
            x = x[:, ::-1].copy()
            y = y[:, ::-1].copy()
        return torch.from_numpy(x).unsqueeze(0), torch.from_numpy(y).unsqueeze(0)

In [ ]:
train_ds = ArcadeBinary('train', train=True)
val_ds = ArcadeBinary('val')
test_ds = ArcadeBinary('test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print('train / val / test images:', len(train_ds), len(val_ds), len(test_ds))

share = [train_ds[i][1].mean().item() for i in range(min(50, len(train_ds)))]
print(f'Foreground share of pixels (first 50 train images): {np.mean(share) * 100:.2f}%')

In [ ]:
import torch.nn as nn


def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = block(1, base)
        self.e2 = block(base, base * 2)
        self.e3 = block(base * 2, base * 4)
        self.e4 = block(base * 4, base * 8)
        self.bott = block(base * 8, base * 16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.d4 = block(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.d3 = block(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.d2 = block(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.d1 = block(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bott(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)


def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    total = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (total + eps)).mean()


bce = nn.BCEWithLogitsLoss()


def loss_fn(logits, target):
    return bce(logits, target) + dice_loss(logits, target)

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tp = fp = fn = tn = 0.0
    dice_per_image = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = (torch.sigmoid(model(x)) > THRESHOLD).float()
        tp += (pred * y).sum().item()
        fp += (pred * (1 - y)).sum().item()
        fn += ((1 - pred) * y).sum().item()
        tn += ((1 - pred) * (1 - y)).sum().item()
        inter = (pred * y).sum(dim=(1, 2, 3))
        total = pred.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
        dice_per_image.extend(((2 * inter + 1e-6) / (total + 1e-6)).cpu().tolist())
    e = 1e-9
    return {
        'dice': 2 * tp / (2 * tp + fp + fn + e),
        'iou': tp / (tp + fp + fn + e),
        'precision': tp / (tp + fp + e),
        'recall_sensitivity': tp / (tp + fn + e),
        'specificity': tn / (tn + fp + e),
        'pixel_accuracy': (tp + tn) / (tp + tn + fp + fn + e),
        'mean_dice_per_image': float(np.mean(dice_per_image)),
    }

In [ ]:
class AttGate(nn.Module):
    def __init__(self, c, inter):
        super().__init__()
        self.wg = nn.Conv2d(c, inter, 1)
        self.wx = nn.Conv2d(c, inter, 1)
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.Sigmoid())
    def forward(self, g, x):
        return x * self.psi(torch.relu(self.wg(g) + self.wx(x)))

class AttUNet(UNet):
    def __init__(self, base=32):
        super().__init__(base)
        self.a4 = AttGate(base*8, base*4)
        self.a3 = AttGate(base*4, base*2)
        self.a2 = AttGate(base*2, base)
        self.a1 = AttGate(base, base//2)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bott(self.pool(e4))
        u = self.u4(b); d4 = self.d4(torch.cat([u, self.a4(u, e4)], 1))
        u = self.u3(d4); d3 = self.d3(torch.cat([u, self.a3(u, e3)], 1))
        u = self.u2(d3); d2 = self.d2(torch.cat([u, self.a2(u, e2)], 1))
        u = self.u1(d2); d1 = self.d1(torch.cat([u, self.a1(u, e1)], 1))
        return self.out(d1)

In [ ]:
model = AttUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
best_dice = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        running += loss.item() * x.size(0)
    train_loss = running / len(train_ds)
    m = evaluate(model, val_loader)
    history.append((epoch, train_loss, m['dice']))
    if m['dice'] > best_dice:
        best_dice = m['dice']
        torch.save(model.state_dict(), f'best_{TASK}_att.pt')
    print(f"epoch {epoch:02d} | train loss {train_loss:.4f} | val dice {m['dice']:.4f} | val recall {m['recall_sensitivity']:.4f}")

print('Best validation Dice:', round(best_dice, 4))

In [ ]:
import json
model.load_state_dict(torch.load(f'best_{TASK}_att.pt', map_location=device))
model.eval()
test_m = evaluate(model, test_loader)
print('Test metrics:', test_m)

results = {'task': TASK, 'model': 'AttUNet',
           'best_val_dice': float(best_dice),
           'test': test_m, 'history': history}
with open(f'results_{TASK}_att.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
for name in ['best_stenosis_att.pt', 'results_stenosis_att.json']:
    shutil.copy(name, '/content/drive/MyDrive/' + name)
print('Saved to Drive')

## Final test results
Run the next cell **once**, only after training is finished. The test images were never used for training or tuning.

In [ ]:
import shutil

model.load_state_dict(torch.load(f'best_{TASK}.pt', map_location=device))
results = evaluate(model, test_loader)

print(f'TEST RESULTS ({TASK})')
for k, v in results.items():
    print(f'{k:22s} {v:.4f}')

with open(f'results_{TASK}.json', 'w') as f:
    json.dump(results, f, indent=2)

for name in [f'best_{TASK}_att.pt', f'results_{TASK}_att.json']:
    shutil.copy(name, '/content/drive/MyDrive/' + name)
print('Saved the model and the results to your Google Drive.')

In [ ]:
model = AttUNet().to(device)
model.load_state_dict(torch.load(f'best_{TASK}_att.pt', map_location=device))
model.eval()
print('val dice ', evaluate(model, val_loader)['dice'])
print('test dice', evaluate(model, test_loader)['dice'])

In [ ]:
import inspect
print('TASK =', TASK, '| THRESHOLD =', THRESHOLD)
print(inspect.getsource(evaluate))

In [ ]:
import matplotlib.pyplot as plt

ep = [h[0] for h in history]
plt.figure(figsize=(6, 4))
plt.plot(ep, [h[1] for h in history], label='train loss')
plt.plot(ep, [h[2] for h in history], label='validation Dice')
plt.xlabel('epoch')
plt.legend()
plt.show()

model.eval()
x, y = next(iter(test_loader))
with torch.no_grad():
    pred = (torch.sigmoid(model(x.to(device))) > THRESHOLD).cpu()

fig, ax = plt.subplots(4, 3, figsize=(9, 12))
for i in range(4):
    ax[i, 0].imshow(x[i, 0], cmap='gray')
    ax[i, 0].set_title('X-ray')
    ax[i, 1].imshow(y[i, 0], cmap='gray')
    ax[i, 1].set_title('Doctor mask')
    ax[i, 2].imshow(pred[i, 0], cmap='gray')
    ax[i, 2].set_title('Model prediction')
for a in ax.ravel():
    a.axis('off')
plt.show()